**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/sennesh-2025"], check=True)
    os.chdir("/content/encore/entries/sennesh-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Sennesh et al. (2025) on dandiset 000253: claims across the six areas

**Paper.** Sennesh E, Westerberg JA, Spencer-Smith J, Bastos AM (2025). Ubiquitous predictive
processing in the spectral domain of sensory cortex. bioRxiv,
[doi:10.1101/2025.07.31.667946](https://doi.org/10.1101/2025.07.31.667946). Claims file
`claims/sennesh-2025.yaml`, written from the preprint's full text and the code at
github.com/BastosLab/epych commit cee2e6f.

**Dandiset.** [DANDI:000253](https://dandiarchive.org/dandiset/000253/0.240923.1441), version
0.240923.1441. The paper cites 0.240503.0152, whose LFP cannot be put on the stimulus clock in 7
of the 14 mice; `v1_laminar.ipynb` shows the differences and gives the reasons for the choice.

**What this notebook tests.** The claims that compare the six areas (V1, LM, RL, AL, PM, AM):
the aperiodic components (Supplementary Figures 1 and 3), the per-mouse gamma and alpha/beta peaks
(Supplementary Figure 2) and the L2/3 gamma power of Figure 5. To fit the time limit, each area is
read only at its kept rows (every fourth aligned channel, counted from the deepest as in the
paper's code, S1), without the median filter over neighbouring channels, and the per-mouse spectra average over those rows. The V1 claims that
need every channel are in `v1_laminar.ipynb`.

| Claim | Tested here |
|---|---|
| dd-aperiodic-no-difference, suppfig1-aperiodic-no-difference-all | yes |
| dd-gamma-peak-frequency-lm, dd-gamma-power-and-peak-only-v1-lm, ssa-gamma-peak-frequency-lower | yes |
| suppfig2-alphabeta-peaks, suppfig2-gamma-height-differences | yes |
| fig5-l23-gamma-by-area, fig5-hierarchy-restriction | yes |
| suppfig3-fastsweep-aperiodic | yes |

In [ ]:
import time
import resource
import zlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import glo

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
PAPER = "sennesh-2025"
OFFS = [0, 1000, 2000, 3000, 4000]     # seed offsets (protocol: the claim's seed and four more)
started = time.time()


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


sess, smeta, pmeta = glo.survey()
mice = sorted(smeta)
conds = {m: glo.conditions(smeta[m][1]) for m in mice}
print(f"dandiset {glo.DANDISET_ID} version {glo.DANDISET_VERSION}: {len(mice)} subjects; survey {time.time() - started:.0f} s")

## Data check

One probe per mouse and area: where a mouse has two probes labelled with one area, the probe with
more channels of that area is used and the other is not used. Areas are aligned across mice as
in S1.

In [ ]:
probe = {}
for x in pmeta:
    if x[5] is None:
        continue
    n = int((x[4].location.map(glo.area_of) == x[5]).sum())
    if (x[0], x[5]) not in probe or n > probe[(x[0], x[5])][1]:
        probe[(x[0], x[5])] = (x, n)
rows = []
align = {}
for area in glo.AREAS:
    ms = [m for m in mice if (m, area) in probe]
    ch = {m: glo.cortical_channels(probe[(m, area)][0][4], area) for m in ms}
    spans, labels, kept, raw = glo.alignment(ch)
    align[area] = (ms, ch, spans, labels, kept)
    kl = [labels[k] for k in kept]
    rows.append({"area": area, "mice": len(ms), "aligned_rows": len(labels), "kept_rows": len(kept),
                 "kept_L2/3": kl.count("L2/3"), "kept_layers": "".join({"L1": "1", "L2/3": "2", "L4": "4", "L5/6": "5"}[k] for k in kl),
                 "missing": [m for m in mice if m not in ms]})
print(pd.DataFrame(rows).to_string(index=False))
print("\ntrials per condition, summed over mice:", {c: sum(len(conds[m][c]) for m in mice) for c in glo.CONDITIONS})

Three mice have no probe labelled AL, so AL has 11 mice; the other areas have 14. In the
`kept_layers` string, 1, 2, 4 and 5 stand for L1, L2/3, L4 and L5/6, from the surface down.

In [ ]:
jobs = [(a, m) for a in glo.AREAS for m in align[a][0]]
t = time.time()
metas = glo.chunk_indexes([(probe[(m, a)][0][2], probe[(m, a)][0][3]) for a, m in jobs], workers=24)
metas = dict(zip(jobs, metas))
print(f"chunk indexes of {len(jobs)} files in {time.time() - t:.0f} s")
clock = pd.DataFrame([{"area": a, "mouse": m, "max_resid_ms": 1000 * glo.lfp_clock(metas[(a, m)])[2],
                       "fs": glo.lfp_clock(metas[(a, m)])[1]} for a, m in jobs])
print("largest clock residual (ms):", clock.max_resid_ms.max(), "; sampling rates", clock.fs.min().round(4), "to", clock.fs.max().round(4))

## Per area: spectrograms, cluster tests and per-mouse spectra

For each area, all mice are read, the S2 spectrograms and S3 aperiodic removal are computed as
in `v1_laminar.ipynb`, the three contrasts are cluster-tested (S4) under both S3 readings, and
the masked L2/3 gamma values of Figure 5 are kept; the per-mouse spectra are kept for the claims
on Supplementary Figures 1 to 3. The four extra seeds are run for S3 reading (a) only, to fit
the time limit.

In [ ]:
freqs, times = glo.FOI, glo.TOI
FIT = {"a": [5, 150], "b": [5, 90]}
psd, n_trials, fig5, fast = {}, {}, {}, {}
for area in glo.AREAS:
    ms, ch, spans, labels, kept = align[area]
    kl = np.array([labels[k] for k in kept])
    t = time.time()
    spec = {c: [] for c in glo.CONDITIONS}
    for m in ms:
        o = glo.light_pass(probe[(m, area)][0][2], metas[(area, m)], ch[m], spans[m], kept, conds[m])
        for c in glo.CONDITIONS:
            spec[c].append(o["spec"][c])
            psd[(area, m, "a", c)] = o["psd"][("a", c)]
            psd[(area, m, "b", c)] = o["psd"][("b", c)]
            n_trials[(area, m, c)] = o["n"][c]
            # Supplementary Figure 3, reading a: per-mouse spectrogram 0-150 ms, mean over time, trials, rows
            fast[(area, m, c)] = o["spec"][c][:, (times >= 0) & (times <= 0.15)].mean((0, 1))   # freq, row
    pf = o["psd_freqs"]
    power = {c: np.concatenate(spec[c]) for c in glo.CONDITIONS}
    del spec
    read_min = (time.time() - t) / 60
    for c in glo.CONDITIONS:                     # reading b of Supplementary Figure 3: pooled, per row
        fast[(area, "pooled", c)] = power[c][:, (times >= 0) & (times <= 0.15)].mean((0, 1))
    for r, rng_ in FIT.items():
        dbs = {}
        for c in glo.CONDITIONS:
            mean = power[c].mean((0, 1))
            ap = np.stack([glo.fooof_aperiodic(freqs, mean[:, k], rng_)[0] for k in range(mean.shape[1])], -1)
            dbs[c] = (10 * (np.log10(power[c]) - ap[None, None])).astype(np.float32)
        for name, (ca, cb) in glo.CONTRASTS.items():
            for off in (OFFS if r == "a" else [0]):
                f_obs, lab, pv = glo.cluster_test(dbs[ca], dbs[cb], np.random.default_rng(seed("fig5-l23-gamma-by-area") + off))
                sig = np.isin(lab, 1 + np.flatnonzero(pv < 0.01))
                diff = dbs[ca].mean(0) - dbs[cb].mean(0)
                masked = np.where(sig, diff, 0.0)
                fig5[(area, name, r, off)] = {"masked_l23": masked[:, :, kl == "L2/3"], "mask_l23": sig[:, :, kl == "L2/3"],
                                              "n_sig_clusters": int((pv < 0.01).sum())}
        del dbs
    del power
    print(f"{area}: {len(ms)} mice read in {read_min:.1f} min; tests done; {(time.time() - started) / 60:.1f} min since start; "
          f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Per-mouse spectra and FOOOF fits

Per mouse, area and condition: the trial-mean power spectrum of each kept row. The aperiodic
claims fit one FOOOF model to the spectrum averaged over rows; the peak claims fit FOOOF per row
and average the oscillatory spectra (dB over the aperiodic fit) over rows (claims file,
dd-gamma-peak-frequency-v1). Readings: (a) 50 ms margins, FOOOF 2 to 90 Hz; (b) 20 ms margins,
FOOOF 2 to 150 Hz.

In [ ]:
SPEC_FIT = {"a": [2, 90], "b": [2, 150]}
ok = pf > 0
fo = pf[ok]
ap_par, osc = {}, {}
for (area, m, r, c), P in psd.items():
    ap_par[(area, m, r, c)] = glo.fooof_aperiodic(fo, P[:, ok].mean(0), SPEC_FIT[r])[1]
    osc[(area, m, r, c)] = np.mean([10 * (np.log10(P[k, ok]) - glo.fooof_aperiodic(fo, P[k, ok], SPEC_FIT[r])[0])
                                    for k in range(P.shape[0])], 0)
print("fits:", len(ap_par), "spectra;", f"{(time.time() - started) / 60:.1f} min since start")


def per_mouse(area, r, c, fn):
    return np.array([fn(osc[(area, m, r, c)]) for m in align[area][0]])


g = (fo >= 30) & (fo <= 90)
peak_hz = lambda y: fo[g][y[g].argmax()]

## dd-aperiodic-no-difference

> Over the whole stimulus, deviance detection gives no significant difference in the aperiodic
> component of the spectrum between oddball and control in any of the six areas.

In [ ]:
CONTR = glo.CONTRASTS


def aperiodic_tests(name, claim, offs=OFFS):
    rows = []
    ca, cb = CONTR[name]
    for area in glo.AREAS:
        ms = align[area][0]
        for r in ["a", "b"]:
            for j, par in enumerate(["offset", "exponent"]):
                x = np.array([ap_par[(area, m, r, ca)][j] for m in ms])
                y = np.array([ap_par[(area, m, r, cb)][j] for m in ms])
                ps = [glo.welch_perm_test(x, y, np.random.default_rng(seed(claim) + off))[1] for off in offs]
                rows.append({"contrast": name, "area": area, "reading": r, "parameter": par, "n": len(ms),
                             "oddball_mean": x.mean(), "control_mean": y.mean(), "diff": x.mean() - y.mean(),
                             "t": stats.ttest_ind(x, y, equal_var=False).statistic, "p": ps[0], "p_min_5seeds": min(ps), "p_max_5seeds": max(ps)})
    return pd.DataFrame(rows)

In [ ]:
apd = aperiodic_tests("DD", "dd-aperiodic-no-difference")
print(apd.round(4).to_string(index=False))
for r in ["a", "b"]:
    d = apd[apd.reading == r]
    print(f"reading {r}: tests with p < 0.05: {int((d.p < 0.05).sum())} of {len(d)}; areas:", sorted(set(d[d.p < 0.05].area)))

**Result: reproduced.** None of the 24 tests (6 areas, offset and exponent, two readings) has
p < 0.05; the smallest p is 0.53 over all seeds. Areas have 11 (AL) to 14 mice.

## suppfig1-aperiodic-no-difference-all

> Over the whole stimulus, the aperiodic component shows no significant difference between
> oddball and control for any of the three contrasts in any of the six areas.

In [ ]:
ap_all = pd.concat([apd] + [aperiodic_tests(n, "suppfig1-aperiodic-no-difference-all") for n in ["GO", "SSA"]])
print(ap_all[ap_all.p < 0.05].round(4).to_string(index=False))
for r in ["a", "b"]:
    d = ap_all[ap_all.reading == r]
    print(f"reading {r}: {int((d.p < 0.05).sum())} of {len(d)} tests with p < 0.05;",
          "contrast-area pairs:", sorted(set(map(tuple, d[d.p < 0.05][["contrast", "area"]].values))))
# Variant: the legend's 2-SEM bands of the aperiodic curve (offset - log10(f^exponent)) across mice.
rows = []
for name, (ca, cb) in CONTR.items():
    for area in glo.AREAS:
        ms = align[area][0]
        cur = {c: np.array([ap_par[(area, m, "a", c)][0] - np.log10(fo ** ap_par[(area, m, "a", c)][1]) for m in ms]) for c in (ca, cb)}
        mu = {c: v.mean(0) for c, v in cur.items()}
        se = {c: v.std(0, ddof=1) / np.sqrt(len(ms)) for c, v in cur.items()}
        sep = np.abs(mu[ca] - mu[cb]) > 2 * (se[ca] + se[cb])
        rows.append({"contrast": name, "area": area, "freqs_bands_apart_2_90Hz": int(sep[(fo >= 2) & (fo <= 90)].sum())})
print(pd.DataFrame(rows).pivot(index="area", columns="contrast", values="freqs_bands_apart_2_90Hz").to_string())

**Result: reproduced.** None of the 72 tests (three contrasts, six areas, two parameters, two
readings) has p < 0.05, and the 2-SEM bands of the aperiodic curves overlap at every frequency
from 2 to 90 Hz in all 18 comparisons.

## dd-gamma-peak-frequency-lm

> In LM the deviance-detection oddball's gamma peak frequency is significantly higher than the
> control's (control 35 Hz, oddball 41 Hz, p < 0.01).

In [ ]:
def peak_table(name, areas, claim, fn=peak_hz):
    ca, cb = CONTR[name]
    rows = []
    for area in areas:
        for r in ["a", "b"]:
            x, y = per_mouse(area, r, ca, fn), per_mouse(area, r, cb, fn)
            ps = [glo.welch_perm_test(x, y, np.random.default_rng(seed(claim) + off))[1] for off in OFFS]
            rows.append({"area": area, "reading": r, "n": len(x), "oddball_mean": x.mean(), "control_mean": y.mean(),
                         "t": stats.ttest_ind(x, y, equal_var=False).statistic, "p": ps[0], "p_min": min(ps), "p_max": max(ps)})
    return pd.DataFrame(rows)

In [ ]:
pl = peak_table("DD", ["LM", "V1"], "dd-gamma-peak-frequency-lm")
pl["control_in_31.5_38.5"] = pl.control_mean.between(31.5, 38.5)
pl["oddball_in_36.9_45.1"] = pl.oddball_mean.between(36.9, 45.1)
print(pl.round(4).to_string(index=False))

**Result: reproduced.** In LM (13 mice) the gamma peak is 40.46 Hz for "XXXY" and 34.00 Hz for
"xxxy" (reading a), both within tolerance, with p = 0.004 (0.002 to 0.006 over five seeds);
reading (b) gives 40.62 and 34.15 Hz, p = 0.004.

## dd-gamma-power-and-peak-only-v1-lm

> After removing the aperiodic component, only V1 (p < 0.05) and LM (p < 0.01) show significant
> increases in both the power and the peak frequency of gamma oscillations for deviance detection.

In [ ]:
ca, cb = CONTR["DD"]
claim = "dd-gamma-power-and-peak-only-v1-lm"
rows = []
for area in glo.AREAS:
    ms = align[area][0]
    for r in ["a", "b"]:
        X = np.array([osc[(area, m, r, ca)] for m in ms])
        Y = np.array([osc[(area, m, r, cb)] for m in ms])
        xm, ym = X[:, g].max(1), Y[:, g].max(1)
        xp, yp = per_mouse(area, r, ca, peak_hz), per_mouse(area, r, cb, peak_hz)
        for off in OFFS:
            rng = np.random.default_rng(seed(claim) + off)
            pb = np.array([glo.welch_perm_test(X[:, i], Y[:, i], rng)[1] if X[:, i].mean() > Y[:, i].mean() else 1.0
                           for i in np.flatnonzero(g)])
            pmax = glo.welch_perm_test(xm, ym, np.random.default_rng(seed(claim) + off + 1))[1] if xm.mean() > ym.mean() else 1.0
            ppk = glo.welch_perm_test(xp, yp, np.random.default_rng(seed(claim) + off + 2))[1] if xp.mean() > yp.mean() else 1.0
            rows.append({"area": area, "reading": r, "seed_offset": off, "power_any_bin_min_p": pb.min(), "power_bins_p<0.05": int((pb < 0.05).sum()),
                         "power_max_p": pmax, "peak_p": ppk, "peak_diff_hz": xp.mean() - yp.mean()})
pw = pd.DataFrame(rows)
for lvl in [0.05, 0.01]:
    pw[f"both_any_bin_{lvl}"] = (pw.power_any_bin_min_p < lvl) & (pw.peak_p < lvl)
    pw[f"both_max_{lvl}"] = (pw.power_max_p < lvl) & (pw.peak_p < lvl)
print(pw[pw.seed_offset == 0].round(4).to_string(index=False))
print("\nRange over the five seeds:")
print(pw.groupby(["area", "reading"])[["power_any_bin_min_p", "power_max_p", "peak_p", "both_any_bin_0.05", "both_any_bin_0.01"]].agg(["min", "max"]).round(4).to_string())
print("The p-values for 'oddball higher' are 1 where the oddball mean is not higher. Seeds: the claim's seed plus the offset, and +1 and +2 for the second and third tests.")

**Result: reproduced, sensitive to the seed.** Under power reading (a) (any gamma bin) with window
(a), V1 passes both tests at 0.05 (power p = 0.002, peak p = 0.010) and LM both at 0.01 (power p =
0.008, peak p = 0.002); no other area passes both at 0.05 (peak p 0.12 or more, or the oddball
peak not higher). Over the five seeds the LM power p runs from 0.006 to 0.012, so under some seeds
LM fails at 0.01; under window (b) or power reading (b) LM does not pass at 0.01. The any-bin power
test is uncorrected and passes at 0.05 in all six areas under window (a) (0.002 to 0.014), so the
area specificity rests on the peak-frequency test.

## ssa-gamma-peak-frequency-lower

> For stimulus-specific adaptation, the "xxxY" oddball (P = 80%) has a significantly lower gamma
> peak frequency than the "yyyy" control in LM (37 against 43 Hz), RL (37 against 45 Hz), AL (36
> against 44 Hz) and AM (38 against 47 Hz).

In [ ]:
ps_ = peak_table("SSA", list(glo.AREAS), "ssa-gamma-peak-frequency-lower")
expect = {"LM": (37, 43, 0.05), "RL": (37, 45, 0.01), "AL": (36, 44, 0.05), "AM": (38, 47, 0.05)}
ps_["named"] = ps_.area.isin(list(expect))
ps_["lower_at_level"] = [(row.oddball_mean < row.control_mean) and row.p < expect.get(row.area, (0, 0, 0.05))[2] for row in ps_.itertuples()]
ps_["means_in_tol"] = [row.area in expect and abs(row.oddball_mean - expect[row.area][0]) <= 0.1 * expect[row.area][0]
                       and abs(row.control_mean - expect[row.area][1]) <= 0.1 * expect[row.area][1] for row in ps_.itertuples()]
print(ps_.round(4).to_string(index=False))
for r in ["a", "b"]:
    d = ps_[(ps_.reading == r) & ps_.named]
    print(f"reading {r}: named areas lower at their level with means in tolerance: {int((d.lower_at_level & d.means_in_tol).sum())} of 4;"
          f" lower at level: {int(d.lower_at_level.sum())} of 4")

**Result: partially reproduced.** The "xxxY" peak is lower than the "yyyy" peak in all four named
areas, with the means within tolerance, but the test passes at the stated level in LM only under
reading (a) (p = 0.018) and in LM and AM under reading (b) (p = 0.006 and 0.042); RL (p = 0.044
and 0.108, against 0.01) and AL (p = 0.074 and 0.080) do not.

## suppfig2-alphabeta-peaks

> In the oscillatory spectra the deviance-detection control has greater alpha/beta power than the
> oddball in all six areas; the global-oddball control has greater alpha/beta power in V1, LM, RL,
> AL and PM; for stimulus-specific adaptation the oddball has greater alpha/beta power in LM and AL.

In [ ]:
def height_cases(cases, lo, hi):
    """cases: (contrast, area, direction) with direction +1 oddball > control, -1 control > oddball, 0 either."""
    sel = (fo >= lo) & (fo <= hi)
    rows = []
    for name, area, d in cases:
        ca, cb = CONTR[name]
        for r in ["a", "b"]:
            X = np.array([osc[(area, m, r, ca)] for m in align[area][0]])
            Y = np.array([osc[(area, m, r, cb)] for m in align[area][0]])
            mx, my = X.mean(0), Y.mean(0)
            # reading (a): at the frequency of the larger condition's mean maximum in the band
            big = mx if mx[sel].max() >= my[sel].max() else my
            i = np.flatnonzero(sel)[big[sel].argmax()]
            va, vb = X[:, i], Y[:, i]
            # reading (b): per-mouse maximum in the band
            wa, wb = X[:, sel].max(1), Y[:, sel].max(1)
            for rd, (a, b) in {"at_peak": (va, vb), "per_mouse_max": (wa, wb)}.items():
                se_a, se_b = a.std(ddof=1) / np.sqrt(len(a)), b.std(ddof=1) / np.sqrt(len(b))
                diff = a.mean() - b.mean()
                sig_max = abs(diff) > 2 * max(se_a, se_b)
                sig_pool = abs(diff) > 2 * np.sqrt(se_a ** 2 + se_b ** 2)
                right = (d == 0) or (np.sign(diff) == d)
                rows.append({"contrast": name, "area": area, "window": r, "reading": rd, "diff_dB": diff,
                             "2xlarger_SEM": 2 * max(se_a, se_b), "holds": bool(sig_max and right),
                             "holds_pooled_SEM": bool(sig_pool and right)})
    return pd.DataFrame(rows)

In [ ]:
cases_ab = [("DD", a, -1) for a in glo.AREAS] + [("GO", a, -1) for a in ["V1", "LM", "RL", "AL", "PM"]] + [("SSA", a, +1) for a in ["LM", "AL"]]
hab = height_cases(cases_ab, 10, 30)
print(hab.round(3).to_string(index=False))
print(hab.groupby(["window", "reading"])[["holds", "holds_pooled_SEM"]].sum().to_string())

**Result: not reproduced.** At most 5 of the 13 named cases differ by more than 2 SEM in the stated
direction (at the peak frequency, either window): the DD control is higher in V1, LM and AL (and
PM under window a), not in RL or AM; no GO case passes; SSA passes in AL and, under window (b), LM.

## suppfig2-gamma-height-differences

> DD and SSA show significant differences in the height of the gamma oscillatory peak in V1, LM, RL,
> AL and AM, and DD in PM as well.

In [ ]:
cases_g = [("DD", a, 0) for a in glo.AREAS] + [("SSA", a, 0) for a in ["V1", "LM", "RL", "AL", "AM"]]
hg = height_cases(cases_g, 30, 90)
print(hg.round(3).to_string(index=False))
print(hg.groupby(["window", "reading"])[["holds", "holds_pooled_SEM"]].sum().to_string())

**Result: reproduced.** Under either window at the peak frequency 10 of the 11 named cases differ
by more than 2 SEM (SSA in RL does not: 0.87 dB against 0.88); with the pooled SEM 10 and 8; the
per-mouse maximum reading gives 6 (window a) and 4 (window b).

## fig5-l23-gamma-by-area

> In L2/3, the time-averaged gamma power difference in the significant clusters is, for deviance
> detection, about 1.5 dB in V1, 1.1 in LM, 1.3 in RL and 0.3 in AL (not significant in PM and AM);
> for the global oddball 0.2 dB in V1, 0.1 in LM and -0.5 in RL (none in AL, PM, AM); for
> stimulus-specific adaptation 0.4 dB in V1, 1.2 in LM, 0.3 in RL, 1.2 in AL, 0.4 in PM and 0.9 in AM.

In [ ]:
EXP5 = {("DD", "V1"): 1.5, ("DD", "LM"): 1.1, ("DD", "RL"): 1.3, ("DD", "AL"): 0.3, ("GO", "V1"): 0.2, ("GO", "LM"): 0.1,
        ("GO", "RL"): -0.5, ("SSA", "V1"): 0.4, ("SSA", "LM"): 1.2, ("SSA", "RL"): 0.3, ("SSA", "AL"): 1.2, ("SSA", "PM"): 0.4,
        ("SSA", "AM"): 0.9}
NONE5 = [("DD", "PM"), ("DD", "AM"), ("GO", "AL"), ("GO", "PM"), ("GO", "AM")]


def l23_value(area, name, r, off, bname, tavg):
    d = fig5[(area, name, r, off)]
    bb = glo.band_bins(freqs, glo.BANDS[bname])
    mk, ms_ = d["masked_l23"][:, bb, :], d["mask_l23"][:, bb, :]
    if tavg == "whole":          # (a) band mean per time and row, then mean over time, then rows
        return float(mk.mean(1).mean(0).mean()), int(ms_.sum())
    return (float(mk[ms_].mean()) if ms_.any() else 0.0), int(ms_.sum())   # (b) significant points only


rows = []
for r in ["a", "b"]:
    for bname in ["gamma", "gamma_code"]:
        for tavg in ["whole", "significant"]:
            for area in glo.AREAS:
                for name in CONTR:
                    v, npts = l23_value(area, name, r, 0, bname, tavg)
                    rows.append({"S3": r, "band": bname, "time_avg": tavg, "contrast": name, "area": area, "value": v, "sig_points": npts})
f5 = pd.DataFrame(rows)

In [ ]:
summ = []
for (r, bname, tavg), d in f5.groupby(["S3", "band", "time_avg"]):
    hit = 0
    for (name, area), e in EXP5.items():
        v = d[(d.contrast == name) & (d.area == area)].value.iloc[0]
        hit += abs(v - e) <= max(0.1 * abs(e), 0.05)
    none_ok = sum(d[(d.contrast == n) & (d.area == a)].sig_points.iloc[0] == 0 for n, a in NONE5)
    signs = sum(np.sign(d[(d.contrast == n) & (d.area == a)].value.iloc[0]) == np.sign(e) for (n, a), e in EXP5.items())
    summ.append({"S3": r, "band": bname, "time_avg": tavg, "in_tolerance_of_13": hit, "signs_match_of_13": signs,
                 "not_significant_cases_empty_of_5": none_ok})
print(pd.DataFrame(summ).to_string(index=False))
print(f5[(f5.S3 == "a") & (f5.band == "gamma")].pivot_table(index=["time_avg", "area"], columns="contrast", values="value").round(3).to_string())
print(f5[(f5.S3 == "a") & (f5.band == "gamma")].pivot_table(index=["area"], columns="contrast", values="sig_points").to_string())
seedrows = []
for off in OFFS:
    for (name, area) in list(EXP5) + NONE5:
        v, npts = l23_value(area, name, "a", off, "gamma", "whole")
        seedrows.append({"seed_offset": off, "case": f"{name}-{area}", "value": round(v, 3), "sig_points": npts})
print(pd.DataFrame(seedrows).pivot(index="case", columns="seed_offset", values="sig_points").to_string())

**Result: not reproduced.** At best 3 of the 13 values are within tolerance (S3 reading a, 30-50 Hz,
whole-window average) and at best 13 of 13 signs match (S3 reading b, 30-50 Hz, with 2 in
tolerance); under 30-90 Hz the whole-window values are 0 of 13 within tolerance. Of the five cases
the paper calls not significant, at most 2 have no masked L2/3 gamma point: DD-AM (156 points),
GO-AL (4) and GO-AM (2) have some, and DD-PM has 143 under one of the five seeds. `v1_laminar.ipynb` repeats the V1
tests with and without the median filter: the V1 values move by at most 0.09 dB, and its unfiltered
variant gives this notebook's V1 values, with one significant GO point fewer (1307 against 1308).

## fig5-hierarchy-restriction

> Deviance-detection and global-oddball gamma modulations in L2/3 are restricted to early areas (V1,
> LM, RL and AL for deviance detection; V1, LM and RL for the global oddball), while stimulus-specific
> adaptation effects appear in all six areas; RL is the one area with a gamma decrease for global
> oddballs.

In [ ]:
want = {"DD": {"V1", "LM", "RL", "AL"}, "GO": {"V1", "LM", "RL"}, "SSA": set(glo.AREAS)}
rows = []
for (r, bname, tavg), d in f5[f5.time_avg == "whole"].groupby(["S3", "band", "time_avg"]):
    got = {n: set(d[(d.contrast == n) & (d.sig_points > 0)].area) for n in CONTR}
    miss = sum(len(want[n] ^ got[n]) for n in CONTR)
    go_rl = d[(d.contrast == "GO") & (d.area == "RL")].value.iloc[0]
    rows.append({"S3": r, "band": bname, **{f"{n}_areas": sorted(got[n]) for n in CONTR}, "set_differences": miss, "GO_RL_value": round(go_rl, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Under reading (a) the areas with masked L2/3 gamma are V1, LM, RL, AL
and AM for DD (AM extra), V1, RL, AL and AM for GO (LM missing, AL and AM extra) and all six for
SSA: four set differences, more than the one allowed. The GO value in RL is negative (-0.167 dB).

## suppfig3-fastsweep-aperiodic

> In the first 150 ms after onset the aperiodic component differs between oddball and control at
> p < 0.05 in DD-LM, GO-AL, GO-PM and SSA-AM, and at p < 0.01 in GO-V1, SSA-V1 and SSA-PM.

In [ ]:
STAR = {("DD", "LM"): 0.05, ("GO", "AL"): 0.05, ("GO", "PM"): 0.05, ("SSA", "AM"): 0.05, ("GO", "V1"): 0.01, ("SSA", "V1"): 0.01, ("SSA", "PM"): 0.01}
claim = "suppfig3-fastsweep-aperiodic"
rows = []
for fitname, rng_ in FIT.items():
    for name, (ca, cb) in CONTR.items():
        for area in glo.AREAS:
            ms = align[area][0]
            # reading (a): per mouse, mean over rows
            pa = {c: np.array([glo.fooof_aperiodic(freqs, fast[(area, m, c)].mean(1), rng_)[1] for m in ms]) for c in (ca, cb)}
            # reading (b): pooled spectrogram, one fit per row, rows as samples
            pb = {c: np.array([glo.fooof_aperiodic(freqs, fast[(area, "pooled", c)][:, k], rng_)[1] for k in range(fast[(area, "pooled", c)].shape[1])]) for c in (ca, cb)}
            for rd, pp in {"a_per_mouse": pa, "b_rows": pb}.items():
                p_par = [[glo.welch_perm_test(pp[ca][:, j], pp[cb][:, j], np.random.default_rng(seed(claim) + off))[1] for j in range(2)] for off in OFFS]
                pmin = [min(x) for x in p_par]
                rows.append({"fit": fitname, "reading": rd, "contrast": name, "area": area, "p_offset": p_par[0][0], "p_exponent": p_par[0][1],
                             "p_min": pmin[0], "p_min_range_5seeds": f"{min(pmin):.3f}-{max(pmin):.3f}", "starred_level": STAR.get((name, area))})
s3 = pd.DataFrame(rows)
s3["starred_pass"] = [(row.contrast, row.area) in STAR and row.p_min < STAR[(row.contrast, row.area)] for row in s3.itertuples()]
s3["unstarred_pass_0.05"] = [(row.contrast, row.area) not in STAR and row.p_min < 0.05 for row in s3.itertuples()]
print(s3.round(4).to_string(index=False))
print(s3.groupby(["fit", "reading"])[["starred_pass", "unstarred_pass_0.05"]].sum().to_string())
print("(7 starred cases, 11 unstarred)")

**Result: not reproduced.** With rows as samples (reading b), 6 of the 7 starred cases pass at
their level under fit (a) (5 to 150 Hz) and 4 under fit (b), but 3 and 5 of the 11 unstarred cases
also pass at 0.05, more than the 2 allowed; with per-mouse spectra (reading a), none of the 7
passes. The partial clause (3 or 4 starred cases) is read with the same bound on unstarred cases,
so fit (b) is not counted as partial.

## Summary

| Claim | Outcome |
|---|---|
| dd-aperiodic-no-difference | reproduced |
| suppfig1-aperiodic-no-difference-all | reproduced |
| dd-gamma-peak-frequency-lm | reproduced |
| dd-gamma-power-and-peak-only-v1-lm | reproduced (sensitive to the seed) |
| ssa-gamma-peak-frequency-lower | partially reproduced |
| suppfig2-alphabeta-peaks | not reproduced |
| suppfig2-gamma-height-differences | reproduced |
| fig5-l23-gamma-by-area | not reproduced |
| fig5-hierarchy-restriction | not reproduced |
| suppfig3-fastsweep-aperiodic | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Each area is read at its kept rows only (every
fourth aligned channel, from the deepest row as in the paper's code), without the median filter,
to fit the time limit; the per-mouse spectra average over those rows. `v1_laminar.ipynb` repeats
the V1 tests with and without the filter; the V1 Figure 5 values move by at most 0.09 dB. LM has
13 mice and AL 11, as the probe labels allow. For Figure 5 the four extra seeds were run for S3
reading (a) only.

**What was found about the dataset.** See `v1_laminar.ipynb`: the probe area labels leave three mice
without an AL probe and one without an LM probe.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")